In [ ]:
from aeon.benchmarking.published_results import (
    load_classification_bake_off_2023_results,
)
from aeon.visualisation import plot_critical_difference
import polars as pl
import numpy as np
from aeon.datasets.tsc_datasets import univariate
from tscglue import utils
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tempfile
import boto3
import polars as pl
from tqdm import tqdm
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
directory = "figures"
Path(directory).mkdir(parents=True, exist_ok=True)

In [ ]:
def load_s3_parquet_cached(
    s3_prefix: str = "s3://tsc-glue/performance-benchmarking/",
    max_workers: int = 16,
) -> pl.DataFrame:

    cache_dir = os.path.join(tempfile.gettempdir(), "tsc-glue-cache")
    os.makedirs(cache_dir, exist_ok=True)

    parsed = urlparse(s3_prefix)
    bucket = parsed.netloc
    prefix = parsed.path.lstrip("/")

    s3 = boto3.client("s3")

    local_files = {
        f for f in os.listdir(cache_dir) if f.endswith(".parquet")
    }

    paginator = s3.get_paginator("list_objects_v2")
    remote_keys = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".parquet"):
                fname = key.rsplit("/", 1)[-1]
                if fname not in local_files:
                    remote_keys.append(key)

    def _download(key):
        fname = key.rsplit("/", 1)[-1]
        s3.download_file(
            bucket,
            key,
            os.path.join(cache_dir, fname),
        )

    if remote_keys:
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(_download, k) for k in remote_keys]
            for _ in tqdm(as_completed(futures), total=len(futures)):
                pass

    local_paths = sorted(
        os.path.join(cache_dir, f)
        for f in os.listdir(cache_dir)
        if f.endswith(".parquet")
    )

    return pl.read_parquet(local_paths)

res_mine = load_s3_parquet_cached()#.filter(pl.col('fold')==0)
res_mine

In [ ]:
res_mine = res_mine.filter(
    (pl.col("model").str.contains("loky-stacker-v5"))==False
)

In [ ]:
results_arr, datasets, classifiers = load_classification_bake_off_2023_results(
    num_resamples=30, as_array=True
)
results_arr.shape

In [ ]:
v = res_mine.filter(pl.col('fold')==5).pivot(values="test_accuracy", index="dataset", on="model").select(['dataset', 'loky-stacker-v7', 'loky-stacker-v6'])
v.with_columns((pl.col('loky-stacker-v7') - pl.col('loky-stacker-v6')).alias('diff')).sort('diff', descending=True)

In [ ]:
pivoted = res_mine.pivot(values="test_accuracy", index="dataset", on="model", aggregate_function='len')
numeric_cols = [c for c in pivoted.columns if c != "dataset"]
sum_row = pl.DataFrame([{"dataset": "all", **{c: pivoted[c].sum() for c in numeric_cols}}]).with_columns([
    pl.col(c).cast(pl.UInt32) for c in numeric_cols
])
result = pl.concat([pivoted, sum_row])
result

In [ ]:
mine_classifiers = res_mine['model'].unique().to_list()

In [ ]:
df_performance = pl.DataFrame(results_arr, schema=classifiers).with_columns(pl.Series("dataset", datasets))
df_performance_mine = res_mine.pivot(values="test_accuracy", index="dataset", on="model", aggregate_function='mean')
df_performance_full = df_performance.join(other=df_performance_mine, on='dataset')

In [ ]:
df_performance_full = df_performance_full#.drop(mine_classifiers)
len(df_performance_full)

In [ ]:
df_performance_full = df_performance_full.drop([col for col in df_performance_full.columns if df_performance_full[col].null_count() > 0])

In [ ]:
drop_models = ['ShapeDTW', 'EE', 'ResNet', 'CNN', 
               'CIF', 'BOSS', 'TSFresh', 'Arsenal', 
               'HC1', 'Hydra', '1NN-DTW', 'ROCKET', 
               'Mini-R', 'WEASEL-D', 'InceptionT', 'Signatures', 'RISE', 'DrCIF', 'TDE', 
               #'loky-stacker-v5-soft-et',
               #'loky-stacker-v5-soft-ridge',	
               #'loky-stacker-v5-soft-rf',	
               #'loky-stacker-v5-r1',
]

In [ ]:
df_performance_full = df_performance_full.drop(drop_models)

In [ ]:
df_performance_full.columns

In [ ]:
clsf = df_performance_full.select([c for c in df_performance_full.columns if c != "dataset"])
plot_critical_difference(clsf.to_numpy(), clsf.columns)

In [ ]:
clsf = df_performance_full.select([
    c for c in df_performance_full.columns if c != "dataset" and (c in classifiers or c == 'loky-stacker-v7-soft-filter-ridge')
])
plot_critical_difference(clsf.to_numpy(), clsf.columns)
plt.savefig(f"{directory}/critical_difference.pdf", bbox_inches='tight', pad_inches=0)


In [ ]:
clsf

In [ ]:
clsf = df_performance_full.select(['loky-stacker-v6-soft-ridge', 'HC2', 'Hydra-MR', 'cBOSS', 'Catch22', 'RDST', 'WEASEL'])
plot_critical_difference(clsf.to_numpy(), clsf.columns)

In [ ]:
clsf = df_performance_full.select(['loky-stacker-v7-soft-filter-ridge', 'HC2', 'Hydra-MR', 'cBOSS', 'Catch22', 'RDST', 'WEASEL'])
plot_critical_difference(clsf.to_numpy(), clsf.columns)

In [ ]:
df_performance_full

In [ ]:
clsf = df_performance_full.rename({'TSCGlue-3-3-26': 'TSCGlue'}).select(['TSCGlue', 'HC2', 'Hydra-MR', 'cBOSS', 'Catch22', 'RDST', 'WEASEL'])
plot_critical_difference(clsf.to_numpy(), clsf.columns)

In [ ]:
v = (df_performance_full['loky-stacker-v6-soft-ridge'] - df_performance_full['HC2']).to_numpy()
(v>0).sum(), (v<0).sum(), (v==0).sum()

In [ ]:
def dataset_stats():

    stats = []
    for dataset in univariate:
        X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
        stats.append(
            {
                "dataset": dataset,
                "n_train": X_train.shape[0],
                "n_test": X_test.shape[0],
                "n_classes": len(np.unique(y_train)),
                "series_length": X_train.shape[2],
            }
        )
    return pl.DataFrame(stats)


stats = dataset_stats()

df_performance_full.join(stats, on='dataset').sort('n_train')

In [ ]:
from aeon.visualisation import create_multi_comparison_matrix

create_multi_comparison_matrix(clsf.to_pandas())
plt.savefig(f"{directory}/multi_comparison_matrix.pdf", bbox_inches='tight', pad_inches=0)

In [ ]:
def plot_acc_diff_vs_size(df, m1, m2):
    data = df.join(stats, on='dataset').with_columns(pl.col('n_train').log().alias('n_train_log')).sort('n_train')
    plt.scatter(data['n_train'], data[m1]-data[m2], s=10)
    plt.xlabel('Training Set Size')
    plt.ylabel(f'Accuracy Difference ({m1} − {m2})')
    plt.xscale('log')
    plt.grid()
    plt.axhline(0, color='black', linewidth=0.8)
    ylim = plt.gca().get_ylim()
    plt.axhspan(0, ylim[1], alpha=0.08, color='green')
    plt.axhspan(ylim[0], 0, alpha=0.08, color='red')
    plt.text(0.98, 0.95, f'{m1} better', transform=plt.gca().transAxes, ha='right', va='top', fontsize=9, color='green')
    plt.text(0.98, 0.05, f'{m2} better', transform=plt.gca().transAxes, ha='right', va='bottom', fontsize=9, color='red')
    plt.ylim(ylim)


In [ ]:
plot_acc_diff_vs_size(df_performance_full, 'loky-stacker-v7-soft-filter-ridge', 'HC2')

In [ ]:
plot_acc_diff_vs_size(df_performance_full, 'loky-stacker-v7-soft-ridge', 'loky-stacker-v7-soft-rf')

In [ ]:
plot_acc_diff_vs_size(df_performance_full, 'loky-stacker-v7-soft-ridge', 'loky-stacker-v7-soft-et')

In [ ]:
plot_acc_diff_vs_size(df_performance_full, 'loky-stacker-v7-soft-rf', 'loky-stacker-v7-soft-et')

In [ ]:
plot_acc_diff_vs_size(df_performance_full, 'loky-stacker-v8-auto-best', 'loky-stacker-v8-base')


In [ ]:
res_mine_f0 = res_mine.filter(pl.col('fold')==0)